# 05 — Streaming: `message/stream` over Server-Sent Events

## Why this notebook exists

In **notebook 04** the client polled `tasks/get` to learn about state changes. Every polling interval that didn't produce news was wasted work, and the client always lagged the server by up to one interval. With many concurrent users on slow tasks, that adds up to a lot of useless requests.

A2A's answer is `message/stream`. Same semantics as `message/send` — *"please do this thing"* — but the server keeps the HTTP connection open and **pushes** events as the work progresses: status updates, artifacts, and a final status that closes the stream.

This notebook builds a streaming researcher, walks through the wire format (Server-Sent Events wrapping JSON-RPC responses), and consumes the stream from a client with no polling at all.

> *Targets A2A spec v0.3.0.*

## What you'll learn

- The Server-Sent Events (SSE) wire format and the `text/event-stream` content type.
- How A2A wraps each SSE event as a complete JSON-RPC 2.0 response.
- The two event kinds: **`TaskStatusUpdateEvent`** (state transitions, with `final: true` on the last one) and **`TaskArtifactUpdateEvent`** (output deliveries).
- How to implement the server side with FastAPI's `StreamingResponse` and a generator.
- How to consume the stream on the client side with `httpx.stream()`.
- Why streaming is strictly better than polling for any non-trivial task — and the one tradeoff (held-open connections) that motivates push notifications in notebook 06.